# Hệ Thống Gợi Ý Việc Làm Thông Minh (Hybrid Recommendation System)

Đây là phiên bản Jupyter Notebook của đồ án tốt nghiệp: **"Hệ thống gợi ý việc làm dựa trên phân tích CV và mô hình PhoBERT"**.

--- 

## 1. Tổng quan Quy trình xử lý (Data Pipeline)

Toàn bộ luồng từ dữ liệu thô đến đặc trưng được tóm tắt qua các bước sau:

*   **Bước 1 – Thu thập**: Crawl từ TopCV/VietnamWorks → CSV (12 trường dữ liệu, tập trung ngành IT).
*   **Bước 2 – Làm sạch**: lowercase → xóa ký tự đặc biệt → chuẩn hóa khoảng trắng → loại bỏ stop words.
*   **Bước 3 – Ghép trường**: title×3 + skills×2 + category×2 + description + requirements + location.
*   **Bước 4 – Vector hóa**: TF-IDF (unigram + bigram, max 5.000 đặc trưng, sublinear TF).
*   **Bước 5 – Tương đồng**: Cosine Similarity → Top-N gợi ý → Cache & Hiển thị kết quả.

---

In [ ]:
# ---------------------------------------------------------------------------
# 1. Khởi tạo môi trường & Cài đặt thư viện
# ---------------------------------------------------------------------------

# Cài đặt các thư viện bổ trợ cho xử lý ngôn ngữ và máy học
!pip install pyvi transformers torch scikit-learn pandas pdfplumber numpy matplotlib seaborn

import os, re, json, unicodedata
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModel
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Thiết lập giao diện biểu đồ theo phong cách học thuật (whitegrid)
plt.rcParams['font.family'] = 'sans-serif'
sns.set_theme(style="whitegrid")
print("Hệ thống đã sẵn sàng.")

## 2. Bước 2: Làm sạch dữ liệu (Data Cleaning)

Quy trình thực hiện: **lowercase → xóa ký tự đặc biệt → chuẩn hóa khoảng trắng → loại stop words**.

Dữ liệu sau khi làm sạch sẽ giúp các thuật toán trọng số tập trung vào các từ khóa kỹ năng (Technical Keywords) thay vì các hư từ vô nghĩa.

In [ ]:
# Danh sách các kỹ năng IT cơ bản phục vụ so khớp từ khóa (Keyword Matching)
SKILL_LIST = [
    "python", "java", "c#", "c++", "golang", "ruby", "php", "nodejs", "react", "reactjs", 
    "vue", "angular", "html", "css", "javascript", "typescript", "docker", "kubernetes", 
    "aws", "azure", "gcp", "mysql", "postgresql", "mongodb", "redis", "nlp", "machine learning",
    "spring boot", "django", "laravel", "flutter", "react native", "swift", "kotlin"
]

def clean_text(text):
    """
    Hàm chuẩn hóa văn bản theo 4 bước yêu cầu:
    1. Lowercase (Chữ thường)
    2. Xóa ký tự đặc biệt bằng biểu thức chính quy (Regex)
    3. Chuẩn hóa Unicode và xóa khoảng trắng dư thừa
    """
    if not isinstance(text, str): return ""
    
    # Bước 2a: Chuyển về chữ thường
    text = text.lower()
    
    # Bước 2b: Giữ lại chữ cái, số, dấu câu cơ bản. Xóa các emoji/icon rác.
    text = re.sub(r"[^\w\s.,;!?-]", " ", text, flags=re.UNICODE)
    
    # Bước 2c: Chuẩn hóa mã Unicode (NFC) tránh lỗi phông tiếng Việt
    text = "".join(ch for ch in text if unicodedata.category(ch)[0] not in ('C',))
    
    # Bước 2d: Thu gọn nhiều khoảng trắng/newline về 1 khoảng trắng duy nhất
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

def normalize_city(city_str):
    """
    Đưa các biến thể địa điểm về định dạng chuẩn để lọc dữ liệu thực tế.
    """
    if not isinstance(city_str, str): return "other"
    city_str = city_str.lower()
    if any(k in city_str for k in ["hà nội", "ha noi", "hanoi"]): return "Hanoi"
    if any(k in city_str for k in ["hồ chí minh", "hcm", "hcmc", "tp hcm", "saigon"]): return "HCM"
    if any(k in city_str for k in ["đà nẵng", "da nang"]): return "Da Nang"
    if "remote" in city_str: return "Remote"
    return "Other"

## 3. Bước 3: Ghép trường (Field Boosting)

Sử dụng công thức tính trọng số tích hợp: 
**title×3 + skills×2 + category×2 + description + requirements + location**.

Mục đích: Tăng cường tầm quan trọng của Tiêu đề và Kỹ năng so với mô tả chi tiết.

In [ ]:
def boost_fields(row):
    """
    Hàm ghép các trường dữ liệu theo trọng số đồ án yêu cầu.
    Nhân bản các trường quan trọng để tăng tần suất xuất hiện (Term Frequency).
    """
    title = str(row.get('job_title', ''))
    skills = str(row.get('skills_json', ''))
    category = str(row.get('category', ''))
    desc = str(row.get('description', ''))
    req = str(row.get('requirements', ''))
    loc = str(row.get('location', ''))
    
    # Thực hiện boosting: title lặp lại 3 lần, skills và category lặp lại 2 lần.
    combined = (title + " ") * 3 + (skills + " ") * 2 + (category + " ") * 2 + desc + " " + req + " " + loc
    
    # Làm sạch văn bản sau khi ghép
    return clean_text(combined)

## 4. Bước 4: Vector hóa (Vectorization & Modeling)

Hệ thống sử dụng mô hình Hybrid kết hợp TF-IDF (Key-matching) và PhoBERT (Semantic-matching).

In [ ]:
# Khởi tạo mô hình Transformer PhoBERT phục vụ Semantic Matching
MODEL_NAME = "vinai/phobert-base"
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

def get_embedding(text):
    """
    Chuyển đổi văn bản sang vector 768 chiều bằng PhoBERT.
    """
    if not isinstance(text, str): text = ""
    inputs = tokenizer(text[:256], return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings.cpu().numpy()[0]

## 5. Phân tích Trực quan Dữ liệu thu thập (Step 1 EDA)

Thống kê tổng quan dựa trên bộ dữ liệu IT tổng hợp (~3.700 jobs).

In [ ]:
# Tải file dữ liệu tổng hợp (Merge từ TopCV và VietnamWorks)
DATA_PATH = "data/TOTAL_IT_JOBS_DATASET.csv"
df_jobs = pd.read_csv(DATA_PATH).dropna(subset=['job_title'])

# Hình 1: Phân bố việc làm IT theo Thành phố lớn
plt.figure(figsize=(10, 6))
df_jobs['city_norm'] = df_jobs['location'].apply(normalize_city)
city_counts = df_jobs['city_norm'].value_counts()
sns.barplot(x=city_counts.index, y=city_counts.values, palette="viridis")
plt.title("Hình 1: Phân bố việc làm IT theo Thành phố", fontsize=12)
plt.ylabel("Số lượng công việc")
plt.show()

In [ ]:
# Hình 2: Top 10 Ngành nghề/Vị trí IT phổ biến nhất
plt.figure(figsize=(12, 6))
top_cats = df_jobs['category'].value_counts().head(10)
sns.barplot(y=top_cats.index, x=top_cats.values, palette="magma")
plt.title("Hình 2: Top 10 Nhóm ngành IT phổ biến nhất", fontsize=12)
plt.show()

In [ ]:
# Hình 3: Top 15 Kỹ năng được yêu cầu nhiều nhất (%) trong Dataset
all_content = " ".join(df_jobs['description'].fillna("").tolist() + df_jobs['job_title'].tolist()).lower()
skill_stats = {skill: all_content.count(skill) for skill in SKILL_LIST}
skill_df = pd.DataFrame(list(skill_stats.items()), columns=['Skill', 'Count']).sort_values('Count', ascending=False)

plt.figure(figsize=(12, 7))
sns.barplot(y=skill_df['Skill'].head(15), x=skill_df['Count'].head(15), palette="rocket")
plt.title("Hình 3: Top 15 Kỹ năng công nghệ hàng đầu", fontsize=12)
plt.show()

## 6. Bước 5: Tính toán Tương đồng và Gợi ý (The Recommender)

Thuật toán: **TF-IDF + PhoBERT Hybrid Scoring**.

In [ ]:
def recommend_jobs(cv_text, df, top_n=5, location_filter=True):
    """
    Hàm phối hợp các thành phần Logic để đưa ra danh sách công việc gợi ý.
    """
    cv_clean = clean_text(cv_text)
    cv_city = normalize_city(cv_text)
    
    # Lọc địa điểm sơ bộ
    if location_filter and cv_city != "Other":
        df_filtered = df[df['location'].apply(normalize_city).isin([cv_city, "Remote"])].copy()
        if len(df_filtered) < 5: df_filtered = df.copy()
    else:
        df_filtered = df.copy()
    
    # Ghép trường tăng cường cho tập dữ liệu
    job_texts = df_filtered.apply(boost_fields, axis=1).tolist()
    
    # Thành phần 1: TF-IDF Similarity
    vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2), sublinear_tf=True)
    tfidf_matrix = vectorizer.fit_transform(job_texts + [cv_clean])
    tfidf_scores = cosine_similarity(tfidf_matrix[-1], tfidf_matrix[:-1])[0]
    
    # Thành phần 2: PhoBERT Semantic Similarity
    cv_emb = get_embedding(cv_clean)
    top_tfidf_idx = np.argsort(tfidf_scores)[-20:][::-1]
    
    final_results = []
    for idx in top_tfidf_idx:
        job_text = job_texts[idx]
        job_emb = get_embedding(job_text)
        bert_score = cosine_similarity([cv_emb], [job_emb])[0][0]
        
        # Trọng số phối hợp (Hybrid Rule)
        hybrid_score = 0.4 * tfidf_scores[idx] + 0.6 * bert_score
        
        job_info = df_filtered.iloc[idx].to_dict()
        job_info['match_score'] = round(hybrid_score * 100, 2)
        final_results.append(job_info)
    
    return sorted(final_results, key=lambda x: x['match_score'], reverse=True)[:top_n]

## 7. Demo Chạy thử hệ thống

Mẫu thử nghiệm với CV về vị trí Backend Engineer.

In [ ]:
test_cv = """
Backend Developer Freelance
Khu vực làm việc: Hà Nội hoặc Remote
Công nghệ sử dụng: Python, Go, Microservices, Kubernetes.
"""

results = recommend_jobs(test_cv, df_jobs)

print(f"--- KẾT QUẢ GỢI Ý (VÙNG: {normalize_city(test_cv).upper()}) ---\n")
for i, res in enumerate(results, 1):
    print(f"{i}. {res['job_title']} - {res['company']}")
    print(f"   Địa điểm: {res['location']} | Điểm phù hợp: {res['match_score']}%\n")